In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("outputs/cleaned_data.csv")

df.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality,revenue
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,4254.50
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,9451.50
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,1819.35
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,1995.92
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,1030.96


In [3]:
df["revenue"] = df["units_sold"] * df["price"]

In [4]:
product_summary = (
    df.groupby("product_id")
      .agg({
          "units_sold":"sum",
          "inventory_level":"mean",
          "revenue":"sum"
      })
      .reset_index()
)

product_summary.head()

,product_id,units_sold,inventory_level,revenue
0,P0001,498061,272.993981,27477692.24
1,P0002,487827,271.146101,26797736.64
2,P0003,493279,274.932695,27057650.52
3,P0004,495501,271.556498,27477122.68
4,P0005,503648,273.041860,27916663.02


In [5]:
product_summary = product_summary.sort_values(
    "revenue",
    ascending=False
)

product_summary["cum_pct"] = (
    product_summary["revenue"].cumsum()
    /
    product_summary["revenue"].sum()
) * 100

In [6]:
def abc_class(x):

    if x <= 80:
        return "A"

    elif x <= 95:
        return "B"

    else:
        return "C"

In [7]:
product_summary["abc_class"] = (
    product_summary["cum_pct"]
    .apply(abc_class)
)

product_summary.head()

,product_id,units_sold,inventory_level,revenue,cum_pct,abc_class
19,P0020,507708,276.798358,28306192.40,5.144440,A
10,P0011,499362,272.807661,28155025.56,10.261406,A
15,P0016,508472,277.505062,28153328.38,15.378063,A
13,P0014,507622,277.892202,28110375.77,20.486915,A
4,P0005,503648,273.041860,27916663.02,25.560560,A


In [9]:
demand_stats = (
    df.groupby("product_id")
      .agg({
          "units_sold":["mean","std"]
      })
)

demand_stats.columns = [
    "avg_daily_demand",
    "std_demand"
]

demand_stats = demand_stats.reset_index()

demand_stats.head()

,product_id,avg_daily_demand,std_demand
0,P0001,136.268399,109.315675
1,P0002,133.468399,106.417492
2,P0003,134.960055,108.380437
3,P0004,135.567989,109.263223
4,P0005,137.796990,106.854998


In [10]:
inventory_df = pd.merge(
    product_summary,
    demand_stats,
    on="product_id"
)

inventory_df.head()

,product_id,units_sold,inventory_level,revenue,cum_pct,abc_class,avg_daily_demand,std_demand
0,P0020,507708,276.798358,28306192.40,5.144440,A,138.907798,110.229782
1,P0011,499362,272.807661,28155025.56,10.261406,A,136.624350,108.245989
2,P0016,508472,277.505062,28153328.38,15.378063,A,139.116826,109.389823
3,P0014,507622,277.892202,28110375.77,20.486915,A,138.884268,110.203837
4,P0005,503648,273.041860,27916663.02,25.560560,A,137.796990,106.854998


In [11]:
service_factor = 1.65

inventory_df["safety_stock"] = (
    service_factor
    *
    inventory_df["std_demand"]
)

In [12]:
lead_time = 7

inventory_df["reorder_point"] = (
    inventory_df["avg_daily_demand"] * lead_time
    +
    inventory_df["safety_stock"]
)

In [13]:
inventory_df["inventory_ratio"] = (
    inventory_df["inventory_level"]
    /
    inventory_df["reorder_point"]
)

In [16]:
q1 = inventory_df["inventory_ratio"].quantile(0.33)
q2 = inventory_df["inventory_ratio"].quantile(0.66)

def classify_risk(x):

    if x <= q1:
        return "HIGH RISK"

    elif x <= q2:
        return "MEDIUM RISK"

    else:
        return "SAFE"

inventory_df["risk_status"] = inventory_df[
    "inventory_ratio"
].apply(classify_risk)

In [17]:
inventory_df[
    [
        "product_id",
        "inventory_level",
        "reorder_point",
        "inventory_ratio",
        "risk_status"
    ]
].head(20)

,product_id,inventory_level,reorder_point,inventory_ratio,risk_status
0,P0020,276.798358,1154.233723,0.239811,HIGH RISK
1,P0011,272.807661,1134.976333,0.240364,HIGH RISK
2,P0016,277.505062,1154.310991,0.240408,HIGH RISK
3,P0014,277.892202,1154.026207,0.240802,MEDIUM RISK
4,P0005,273.041860,1140.889679,0.239324,HIGH RISK
5,P0013,275.111354,1137.223966,0.241915,MEDIUM RISK
6,P0015,276.408208,1152.900306,0.239750,HIGH RISK
7,P0009,275.324761,1144.012744,0.240666,HIGH RISK
8,P0007,274.713269,1138.625759,0.241267,MEDIUM RISK
9,P0001,272.993981,1134.249659,0.240682,MEDIUM RISK


In [18]:
inventory_df["risk_status"].value_counts()

risk_status
HIGH RISK      7
SAFE           7
MEDIUM RISK    6
Name: count, dtype: int64

In [19]:
inventory_df["inventory_turnover"] = (
    inventory_df["avg_daily_demand"]
    /
    inventory_df["inventory_level"]
)

In [20]:
inventory_df[
    [
        "product_id",
        "abc_class",
        "risk_status",
        "inventory_turnover"
    ]
].head()

,product_id,abc_class,risk_status,inventory_turnover
0,P0020,A,HIGH RISK,0.501838
1,P0011,A,HIGH RISK,0.500808
2,P0016,A,HIGH RISK,0.501313
3,P0014,A,MEDIUM RISK,0.499777
4,P0005,A,HIGH RISK,0.504673


In [22]:
inventory_df.to_csv(
    "outputs/inventory_analysis.csv",
    index=False
)

print("Inventory analysis saved")

Inventory analysis saved
